In [7]:
import io
import os
import boto3
import numpy as np
from PIL import Image
from pathlib import Path
from dotenv import load_dotenv
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
import mlflow
import mlflow.pytorch
from sklearn.model_selection import train_test_split

In [2]:
load_dotenv(Path("../.env"))

s3 = boto3.client(
    "s3",
    endpoint_url=os.getenv("AISTOR_ENDPOINT"),
    aws_access_key_id=os.getenv("AISTOR_ACCESS_KEY"),
    aws_secret_access_key=os.getenv("AISTOR_SECRET_KEY")
)

In [5]:
mlflow.set_tracking_uri(os.getenv("MLFLOW_URI"))
mlflow.set_experiment("fish-classifier")

2026/05/27 17:04:22 INFO mlflow.tracking.fluent: Experiment with name 'fish-classifier' does not exist. Creating a new experiment.


<Experiment: artifact_location='/mlflow/artifacts/1', creation_time=1779926661948, experiment_id='1', last_update_time=1779926661948, lifecycle_stage='active', name='fish-classifier', tags={}, trace_location=None, workspace='default'>

In [6]:
BUCKET = "fish-classifier"

classes = [
    "Black Sea Sprat",
    "Gilt-Head Bream",
    "Hourse Mackerel",
    "Red Mullet",
    "Red Sea Bream",
    "Sea Bass",
    "Shrimp",
    "Striped Red Mullet",
    "Trout"
]

class_to_idx = {cls: idx for idx, cls in enumerate(classes)}
idx_to_class = {idx: cls for cls, idx in class_to_idx.items()}

In [8]:
class FishDataset(Dataset):
    def __init__(self, keys, class_to_idx, transform=None):
        self.keys = keys
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, idx):
        key = self.keys[idx]
        class_name = key.split('/')[1]
        label = self.class_to_idx[class_name]

        response = s3.get_object(Bucket=BUCKET, Key=key)
        img = Image.open(io.BytesIO(response['Body'].read())).convert('RGB')

        if self.transform:
            img = self.transform(img)

        return img, label

In [9]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [10]:
all_keys = []
paginator = s3.get_paginator('list_objects_v2')
for page in paginator.paginate(Bucket=BUCKET, Prefix="raw/"):
    for obj in page.get('Contents', []):
        all_keys.append(obj['Key'])

train_keys, temp_keys = train_test_split(all_keys, test_size=0.3, random_state=42)
val_keys, test_keys = train_test_split(temp_keys, test_size=0.5, random_state=42)

print(f"Train: {len(train_keys)} | Val: {len(val_keys)} | Test: {len(test_keys)}")

Train: 6300 | Val: 1350 | Test: 1350


In [11]:
train_dataset = FishDataset(train_keys, class_to_idx, transform=train_transforms)
val_dataset = FishDataset(val_keys, class_to_idx, transform=val_transforms)
test_dataset = FishDataset(test_keys, class_to_idx, transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

Train batches: 197
Val batches: 43
Test batches: 43


In [12]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Replace final layer with our 9-class classifier
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(classes))

model = model.to(device)
print("Model ready")

Using device: mps
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /Users/bv/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|████████████████████████████████████████████████████████████████████████████████| 20.5M/20.5M [00:00<00:00, 57.7MB/s]


Model ready


In [13]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=0.001)

print("Loss and optimizer ready")

Loss and optimizer ready


In [18]:
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=10):
    with mlflow.start_run(run_name="efficientnet-b0-lr0.001"):
        
        # Log hyperparameters
        mlflow.log_param("model", "efficientnet_b0")
        mlflow.log_param("learning_rate", 0.001)
        mlflow.log_param("batch_size", 32)
        mlflow.log_param("epochs", epochs)
        mlflow.log_param("optimizer", "adam")

        for epoch in range(epochs):
            # Training
            model.train()
            train_loss = 0
            correct = 0
            total = 0

            for images, labels in train_loader:
                images, labels = images.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                train_loss += loss.item()
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

            train_acc = correct / total
            train_loss = train_loss / len(train_loader)

            # Validation
            model.eval()
            val_loss = 0
            val_correct = 0
            val_total = 0

            with torch.no_grad():
                for images, labels in val_loader:
                    images, labels = images.to(device), labels.to(device)
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                    val_loss += loss.item()
                    _, predicted = outputs.max(1)
                    val_total += labels.size(0)
                    val_correct += predicted.eq(labels).sum().item()

            val_acc = val_correct / val_total
            val_loss = val_loss / len(val_loader)

            # Log to MLflow
            mlflow.log_metric("train_loss", train_loss, step=epoch)
            mlflow.log_metric("train_accuracy", train_acc, step=epoch)
            mlflow.log_metric("val_loss", val_loss, step=epoch)
            mlflow.log_metric("val_accuracy", val_acc, step=epoch)

            print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

In [ ]:
train_model(model, train_loader, val_loader, criterion, optimizer, epochs=10)

In [16]:
# Save model locally
torch.save(model.state_dict(), "../models/efficientnet_b0_fish.pth")
print("Model saved to models/")

Model saved to models/
